In [1]:
# Cell 1: Connect to database (read-only — validation never writes)
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.db import get_connection

con = get_connection(read_only=True)
print("Connected (read-only)")

Connected (read-only)


In [2]:
# Cell 2: Row counts — confirm all tables populated
tables = [
    "provider", "claim_header", "claim_line",
    "remittance_835", "encounter_context",
    "denial_forensics", "appeal_outcomes"
]

print("TABLE ROW COUNTS")
print("-" * 35)
for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    status = "✓" if count > 0 else "✗ EMPTY"
    print(f"  {table:<25} {count:>8,}  {status}")

TABLE ROW COUNTS
-----------------------------------
  provider                         5  ✓
  claim_header                25,000  ✓
  claim_line                  69,574  ✓
  remittance_835              69,574  ✓
  encounter_context           25,000  ✓
  denial_forensics            10,089  ✓
  appeal_outcomes              9,641  ✓


In [3]:
# Cell 3: Financial constraint — billed = paid + patient_resp + adjustment
# This must hold for every row in remittance_835

violations = con.execute("""
    SELECT COUNT(*) 
    FROM remittance_835
    WHERE ABS(
        (paid_amount + patient_responsibility + adjustment_amount) -
        (SELECT billed_amount FROM claim_line 
         WHERE claim_line.claim_line_id = remittance_835.claim_line_id)
    ) >= 0.01
""").fetchone()[0]

print(f"Financial constraint violations: {violations}")
print("✓ All rows balanced" if violations == 0 else "✗ VIOLATIONS FOUND — check generate_remittance.py")

Financial constraint violations: 0
✓ All rows balanced


In [4]:
# Cell 4: Denial rate and CARC distribution
# Denial rate should be non-uniform — not 0% or 100%
# CARC distribution should show multiple codes

total  = con.execute("SELECT COUNT(*) FROM remittance_835").fetchone()[0]
denied = con.execute("SELECT COUNT(*) FROM remittance_835 WHERE paid_amount = 0").fetchone()[0]
denial_rate = denied / total

print(f"Overall denial rate: {denial_rate:.1%}")
print(f"  ({denied:,} denied out of {total:,} lines)")

if denial_rate < 0.03:
    print("⚠ Denial rate low — persona logic may not be firing")
elif denial_rate > 0.60:
    print("⚠ Denial rate very high — check adjudication logic")
else:
    print("✓ Denial rate looks realistic")

print("\nCARC code distribution:")
carc_dist = con.execute("""
    SELECT carc_code, COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM remittance_835
    GROUP BY carc_code
    ORDER BY count DESC
""").df()
print(carc_dist.to_string(index=False))

Overall denial rate: 15.3%
  (10,678 denied out of 69,574 lines)
✓ Denial rate looks realistic

CARC code distribution:
carc_code  count  pct
    CO-45  59485 85.5
     CO-4   3276  4.7
    CO-16   1591  2.3
    CO-B7   1505  2.2
    CO-57   1281  1.8
    CO-50    759  1.1
    CO-97    634  0.9
    CO-22    595  0.9
    CO-29    448  0.6


In [5]:
# Cell 5: Persona pattern check
# Each provider persona should produce a distinct denial fingerprint
# Internal medicine → elevated CO-29 (timely filing)
# Radiology → elevated CO-57 (prior auth)
# Emergency → elevated CO-4 (coding errors)
# Behavioral health → CO-B7 (credentialing)

print("Denial rate by provider persona:")
persona_denials = con.execute("""
    SELECT 
        p.persona_type,
        COUNT(r.remittance_id) as total_lines,
        SUM(CASE WHEN r.paid_amount = 0 THEN 1 ELSE 0 END) as denied_lines,
        ROUND(SUM(CASE WHEN r.paid_amount = 0 THEN 1 ELSE 0 END) * 100.0 
              / COUNT(r.remittance_id), 1) as denial_pct
    FROM remittance_835 r
    JOIN claim_header ch ON r.claim_id = ch.claim_id
    JOIN provider p ON ch.provider_id = p.provider_id
    GROUP BY p.persona_type
    ORDER BY denial_pct DESC
""").df()
print(persona_denials.to_string(index=False))

print("\nTop CARC per persona:")
carc_by_persona = con.execute("""
    SELECT 
        p.persona_type,
        r.carc_code,
        COUNT(*) as count
    FROM remittance_835 r
    JOIN claim_header ch ON r.claim_id = ch.claim_id
    JOIN provider p ON ch.provider_id = p.provider_id
    WHERE r.paid_amount = 0
      AND r.carc_code NOT IN ('CO-45', 'PR-1')
    GROUP BY p.persona_type, r.carc_code
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY p.persona_type ORDER BY COUNT(*) DESC
    ) = 1
    ORDER BY count DESC
""").df()
print(carc_by_persona.to_string(index=False))

Denial rate by provider persona:
             persona_type  total_lines  denied_lines  denial_pct
     credentialing_errors        13829        3305.0        23.9
            coding_errors        13994        2230.0        15.9
medical_necessity_denials        13755        1872.0        13.6
          prior_auth_miss        13760        1636.0        11.9
      slow_charge_capture        14236        1635.0        11.5

Top CARC per persona:
             persona_type carc_code  count
     credentialing_errors     CO-B7   1505
            coding_errors      CO-4   1228
          prior_auth_miss     CO-57    681
medical_necessity_denials     CO-57    600
      slow_charge_capture      CO-4    451


In [6]:
# Cell 6: Upstream failure nodes — should have shape, not all "unknown"
print("Upstream failure node distribution:")
nodes = con.execute("""
    SELECT 
        upstream_failure_node,
        COUNT(*) as denial_count,
        ROUND(SUM(dollars_at_risk), 2) as total_at_risk,
        ROUND(AVG(recovery_probability), 3) as avg_recovery_prob
    FROM denial_forensics
    GROUP BY upstream_failure_node
    ORDER BY total_at_risk DESC
""").df()
print(nodes.to_string(index=False))

unknown_pct = con.execute("""
    SELECT ROUND(
        SUM(CASE WHEN upstream_failure_node = 'unknown' THEN 1 ELSE 0 END) * 100.0
        / COUNT(*), 1
    )
    FROM denial_forensics
""").fetchone()[0]

if unknown_pct > 20:
    print(f"\n⚠ {unknown_pct}% of denials classified as 'unknown' — add CARC codes to CARC_CLASSIFICATION")
else:
    print(f"\n✓ Unknown classification rate: {unknown_pct}%")

Upstream failure node distribution:
upstream_failure_node  denial_count  total_at_risk  avg_recovery_prob
  prior_authorization          2040     4890631.54              0.476
               coding          3910     4818146.82              0.692
              billing          1591     1680743.15              0.720
   remittance_posting           595      765555.38              0.550
        credentialing          1505      212264.86              0.150
       charge_capture           448       80433.83              0.100

✓ Unknown classification rate: 0.0%


In [7]:
# Cell 7: Null check on critical fields
print("Null check on critical fields:")
print("-" * 45)

checks = [
    ("claim_header",    "claim_id"),
    ("claim_header",    "payer_id"),
    ("claim_line",      "cpt_code"),
    ("claim_line",      "icd10_primary"),
    ("remittance_835",  "carc_code"),
    ("encounter_context", "prior_auth_required"),
    ("denial_forensics",  "upstream_failure_node"),
    ("denial_forensics",  "priority_score"),
]

all_clean = True
for table, column in checks:
    nulls = con.execute(
        f"SELECT COUNT(*) FROM {table} WHERE {column} IS NULL"
    ).fetchone()[0]
    status = "✓" if nulls == 0 else f"✗ {nulls:,} nulls"
    if nulls > 0:
        all_clean = False
    print(f"  {table}.{column:<30} {status}")

print("\n✓ All critical fields clean" if all_clean else "\n⚠ Nulls found — investigate before analysis")

Null check on critical fields:
---------------------------------------------
  claim_header.claim_id                       ✓
  claim_header.payer_id                       ✓
  claim_line.cpt_code                       ✓
  claim_line.icd10_primary                  ✓
  remittance_835.carc_code                      ✓
  encounter_context.prior_auth_required            ✓
  denial_forensics.upstream_failure_node          ✓
  denial_forensics.priority_score                 ✓

✓ All critical fields clean


In [8]:
# Cell 8: Final validation verdict
print("=" * 45)
print("VALIDATION SUMMARY")
print("=" * 45)

total_claims    = con.execute("SELECT COUNT(*) FROM claim_header").fetchone()[0]
total_lines     = con.execute("SELECT COUNT(*) FROM claim_line").fetchone()[0]
total_denied    = con.execute("SELECT COUNT(*) FROM denial_forensics").fetchone()[0]
total_at_risk   = con.execute("SELECT SUM(dollars_at_risk) FROM denial_forensics").fetchone()[0]
total_recovered = con.execute("SELECT SUM(dollars_recovered) FROM appeal_outcomes").fetchone()[0]

print(f"  Claims:              {total_claims:>8,}")
print(f"  Claim lines:         {total_lines:>8,}")
print(f"  Denied lines:        {total_denied:>8,}")
print(f"  Total at risk:       ${total_at_risk:>11,.2f}")
print(f"  Simulated recovery:  ${total_recovered:>11,.2f}")
print(f"  Recovery rate:       {total_recovered/total_at_risk:.1%}")
print()
print("Database is validated and ready for forensic analysis.")
print("Open notebook 03 to begin the investigation.")

con.close()

VALIDATION SUMMARY
  Claims:                25,000
  Claim lines:           69,574
  Denied lines:          10,089
  Total at risk:       $12,447,775.58
  Simulated recovery:  $6,557,399.28
  Recovery rate:       52.7%

Database is validated and ready for forensic analysis.
Open notebook 03 to begin the investigation.
